[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-06-ml-cicd-patterns.ipynb#scrollTo=aa11bb22)

---
# Day 6 · ML-Specific Patterns — Training, Evaluation, and Model Registry
**certified-journeys / github-actions-certified** · Day 6 · ML CI/CD

> **Goal for today:** Write GitHub Actions workflows that train a sklearn model on push, upload the artifact to GitHub Releases, enforce an accuracy quality gate, and post evaluation results as a PR comment using `actions/github-script`.


In [ ]:
%pip install -q pyyaml scikit-learn


## Step 1 · The ML CI/CD Pattern

Traditional CI runs tests and ships code. ML CI/CD adds three extra concerns:

| Stage | What it does | Failure mode |
|---|---|---|
| **Train** | Fit model, save `.pkl` artifact | Import error, OOM |
| **Evaluate** | Score on held-out set, compare to baseline | Accuracy below threshold |
| **Register** | Upload artifact to model registry | Publish fails |
| **Notify** | Post results to PR comment | Cosmetic only |

The key insight: **evaluation is a quality gate** — if accuracy drops below the threshold, the workflow fails and the model is never registered. The PR shows why.


In [ ]:
import yaml

# Conceptual pipeline structure — visualise the job graph
pipeline_jobs = {
    "test":     {"depends_on": None,     "gate": "pytest passes"},
    "train":    {"depends_on": "test",   "gate": "model.pkl produced"},
    "evaluate": {"depends_on": "train",  "gate": "accuracy >= 0.90"},
    "release":  {"depends_on": "evaluate", "gate": "GitHub Release created"},
    "comment":  {"depends_on": "evaluate", "gate": "PR comment posted"},
}

print("ML CI/CD Job Graph")
print("==================")
for job, info in pipeline_jobs.items():
    dep = info["depends_on"]
    arrow = f"  ← needs: {dep}" if dep else "  (trigger job)"
    print(f"  {job:12s} {arrow}")
    print(f"  {'':12s}  quality gate: {info['gate']}")
    print()


**What just happened?**

- We mapped the full ML CI/CD graph: test → train → evaluate → release/comment.
- Each job has a **quality gate** — if it fails, downstream jobs don't run.
- `release` and `comment` are siblings that both depend on `evaluate` — they run in parallel after evaluation passes.


## Step 2 · Training a sklearn Model Locally (Simulating the CI Step)

In CI, the train step runs `python train.py`. Let's build and test that script locally using the iris dataset — no external data source needed.


In [ ]:
import pickle
import pathlib
import json
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Load and split data — iris is always available, no internet needed
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train a simple RandomForest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Confusion matrix:\n{cm}")

# Save model artifact — this is what the CI train step produces
model_path = pathlib.Path("/tmp/model.pkl")
with open(model_path, "wb") as f:
    pickle.dump(model, f)

print(f"\nModel saved to {model_path} ({model_path.stat().st_size} bytes)")

# Save evaluation results to JSON — shared between evaluate and comment jobs
results = {
    "accuracy": round(accuracy, 4),
    "confusion_matrix": cm.tolist(),
    "threshold": 0.90,
    "passed": accuracy >= 0.90,
}
results_path = pathlib.Path("/tmp/eval_results.json")
results_path.write_text(json.dumps(results, indent=2))
print(f"Results saved to {results_path}")


**What just happened?**

- We trained a RandomForest on iris — in CI this is `python train.py` which produces `model.pkl`.
- `eval_results.json` stores the accuracy and confusion matrix — this is what the evaluate and comment jobs consume.
- **The model artifact and results file are passed between jobs using `actions/upload-artifact` and `actions/download-artifact`** — we'll wire this up in the next step.


## Step 3 · Passing Artifacts Between Jobs

Jobs run on separate ephemeral runners — they share **nothing** by default. `actions/upload-artifact` and `actions/download-artifact` are the bridge:

| Step | Action | What it does |
|---|---|---|
| Train job end | `upload-artifact` | Uploads `model.pkl` to Actions artifact store |
| Evaluate job start | `download-artifact` | Downloads `model.pkl` to the evaluate runner |
| Release job start | `download-artifact` | Downloads same `model.pkl` for GitHub Releases upload |

**Critical:** both download steps must use the **exact same `name:`** as the upload step.


In [ ]:
# Workflow with artifact passing between train, evaluate, and release jobs
artifact_workflow = {
    "name": "ML Train + Evaluate + Release",
    "on": {"push": {"branches": ["main"]}},
    "jobs": {
        "train": {
            "runs-on": "ubuntu-latest",
            "permissions": {"contents": "read"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {"uses": "actions/setup-python@v5", "with": {"python-version": "3.12"}},
                {"run": "pip install -r requirements.txt"},
                {"name": "Train model", "run": "python train.py --output model.pkl"},
                {
                    # Upload artifact — 'name' must match exactly in download steps
                    "name": "Upload model artifact",
                    "uses": "actions/upload-artifact@v4",
                    "with": {
                        "name": "trained-model",       # key used by all download steps
                        "path": "model.pkl",
                        "retention-days": 7,           # auto-delete after 7 days
                    },
                },
            ],
        },
        "evaluate": {
            "runs-on": "ubuntu-latest",
            "needs": "train",               # wait for train to finish
            "permissions": {"contents": "read"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {"uses": "actions/setup-python@v5", "with": {"python-version": "3.12"}},
                {"run": "pip install -r requirements.txt"},
                {
                    # Download the exact same artifact the train job uploaded
                    "name": "Download model artifact",
                    "uses": "actions/download-artifact@v4",
                    "with": {"name": "trained-model"},
                },
                {
                    "name": "Evaluate model",
                    "run": "python evaluate.py --model model.pkl --threshold 0.90 --output eval_results.json",
                },
                {
                    # Upload evaluation results so the comment job can read them
                    "name": "Upload evaluation results",
                    "uses": "actions/upload-artifact@v4",
                    "with": {"name": "eval-results", "path": "eval_results.json"},
                },
            ],
        },
        "release": {
            "runs-on": "ubuntu-latest",
            "needs": "evaluate",
            # Release job needs write access to create releases
            "permissions": {"contents": "write"},
            "steps": [
                {"uses": "actions/checkout@v4"},
                {
                    "name": "Download model artifact",
                    "uses": "actions/download-artifact@v4",
                    "with": {"name": "trained-model"},
                },
                {
                    "name": "Create GitHub Release with model",
                    # softprops/action-gh-release — pin to SHA in production
                    "uses": "softprops/action-gh-release@v2",
                    "with": {
                        "tag_name": "model-${{ github.sha }}",
                        "name": "Model ${{ github.sha }}",
                        "files": "model.pkl",
                        "generate_release_notes": True,
                    },
                },
            ],
        },
    },
}

print(yaml.dump(artifact_workflow, sort_keys=False))


**What just happened?**

- `actions/upload-artifact` in the train job writes `model.pkl` to GitHub's artifact store under the name `trained-model`.
- Both `evaluate` and `release` jobs download it by that same name — they get identical files.
- **`retention-days: 7`** prevents artifact storage costs from accumulating; production pipelines often set this to 30.
- The evaluate job uploads `eval_results.json` as a separate artifact so the comment job can read it too.


## Step 4 · The Accuracy Quality Gate

The evaluate script must **fail with a non-zero exit code** if accuracy is below the threshold. That signals Actions to mark the job as failed and block the release job.


In [ ]:
# This is what evaluate.py looks like — the quality gate script
evaluate_script = '''\
#!/usr/bin/env python
"""evaluate.py — Load a model, score it, fail if below threshold."""
import sys
import json
import pickle
import argparse
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--threshold", type=float, default=0.90)
    parser.add_argument("--output", default="eval_results.json")
    args = parser.parse_args()

    # Load model
    with open(args.model, "rb") as f:
        model = pickle.load(f)

    # Held-out test set (same split as training so results are reproducible)
    X, y = load_iris(return_X_y=True)
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    results = {
        "accuracy": round(float(accuracy), 4),
        "threshold": args.threshold,
        "passed": accuracy >= args.threshold,
        "confusion_matrix": cm.tolist(),
    }

    with open(args.output, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Accuracy: {accuracy:.4f}  Threshold: {args.threshold}")
    print(json.dumps(results, indent=2))

    if not results["passed"]:
        print(f"FAIL: accuracy {accuracy:.4f} < threshold {args.threshold}", file=sys.stderr)
        sys.exit(1)   # Non-zero exit → Actions marks the job as FAILED

    print("PASS: accuracy meets threshold")

if __name__ == "__main__":
    main()
'''

print(evaluate_script)


**What just happened?**

- `sys.exit(1)` is the key line — a non-zero exit code makes Actions mark the job as failed.
- Because `release` has `needs: evaluate`, it will be **skipped** (not run) if evaluate fails.
- The results JSON is always written before the exit check — so the comment job can still read it even when the gate fails.


## Step 5 · Running the Quality Gate Locally

Test the gate logic before committing — we'll simulate both pass and fail scenarios.


In [ ]:
import sys
import json
import pickle
import pathlib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


def run_quality_gate(model_path: str, threshold: float) -> dict:
    """Replicate evaluate.py logic — returns results dict."""
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    X, y = load_iris(return_X_y=True)
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    results = {
        "accuracy": round(float(accuracy), 4),
        "threshold": threshold,
        "passed": accuracy >= threshold,
        "confusion_matrix": cm.tolist(),
    }
    return results


# Test with a threshold the model easily passes
results_pass = run_quality_gate("/tmp/model.pkl", threshold=0.90)
print("=== Threshold 0.90 ===")
print(json.dumps(results_pass, indent=2))
print(f"Gate result: {'PASS ✓' if results_pass['passed'] else 'FAIL ✗'}\n")

# Test with an impossible threshold (demonstrates failure path)
results_fail = run_quality_gate("/tmp/model.pkl", threshold=0.999)
print("=== Threshold 0.999 (intentionally fails) ===")
print(json.dumps(results_fail, indent=2))
print(f"Gate result: {'PASS ✓' if results_fail['passed'] else 'FAIL ✗'}")
print("In CI: sys.exit(1) would be called → job marked FAILED → release job skipped")


**What just happened?**

- The quality gate ran against our saved model — with `threshold=0.90` the RF on iris passes easily.
- With `threshold=0.999` it fails — demonstrating what happens when a model regresses.
- In the real CI evaluate job, `sys.exit(1)` would cause Actions to mark the step red and skip `release`.


## Step 6 · Posting a PR Comment with `actions/github-script`

`actions/github-script` lets you call the GitHub API using JavaScript inside a workflow step. The most common ML use case: post a comment to the PR that triggered the run, showing accuracy and the confusion matrix as a markdown table.


In [ ]:
# The PR comment step — uses actions/github-script with inline JS
comment_step = {
    "name": "Post PR comment with evaluation results",
    "uses": "actions/github-script@v7",
    "with": {
        "script": """
const fs = require('fs');

// Read the evaluation results artifact (downloaded in a previous step)
const results = JSON.parse(fs.readFileSync('eval_results.json', 'utf8'));
const { accuracy, threshold, passed, confusion_matrix } = results;

// Build confusion matrix as a markdown table
const labels = ['setosa', 'versicolor', 'virginica'];
const header = '| | ' + labels.join(' | ') + ' |';
const sep    = '|---|' + labels.map(() => '---').join('|') + '|';
const rows   = confusion_matrix.map((row, i) =>
  `| **${labels[i]}** | ${row.join(' | ')} |`
).join('\\n');

const status = passed ? '✅ PASSED' : '❌ FAILED';
const body = [
  `## 🤖 Model Evaluation Results`,
  ``,
  `| Metric | Value |`,
  `|---|---|`,
  `| Accuracy | ${(accuracy * 100).toFixed(2)}% |`,
  `| Threshold | ${(threshold * 100).toFixed(0)}% |`,
  `| Status | ${status} |`,
  ``,
  `### Confusion Matrix`,
  ``,
  header, sep, rows,
  ``,
  passed
    ? `Model has been uploaded to [GitHub Releases](../../releases).`
    : `⚠️ Model did NOT meet the accuracy threshold — release skipped.`,
].join('\\n');

await github.rest.issues.createComment({
  owner: context.repo.owner,
  repo: context.repo.repo,
  issue_number: context.issue.number,
  body,
});
"""
    },
}

print(yaml.dump(comment_step, sort_keys=False))


**What just happened?**

- `actions/github-script` provides `github` (Octokit client) and `context` (run metadata) as globals.
- `context.issue.number` is the PR number — only populated on `pull_request` event triggers.
- The confusion matrix is rendered as a Markdown table inline in the comment body.
- The comment is posted even if the quality gate fails (because the comment job checks `needs.evaluate.result` independently).


## Step 7 · Rendering the Confusion Matrix Markdown Locally

Let's generate and preview the PR comment body in Python before wiring it into the workflow.


In [ ]:
import json


def render_pr_comment(eval_results: dict, class_names: list[str]) -> str:
    """Generate the markdown body for the PR comment."""
    accuracy = eval_results["accuracy"]
    threshold = eval_results["threshold"]
    passed = eval_results["passed"]
    cm = eval_results["confusion_matrix"]

    status = "✅ PASSED" if passed else "❌ FAILED"

    # Build confusion matrix table
    header = "| | " + " | ".join(f"pred:{c}" for c in class_names) + " |"
    sep    = "|---" * (len(class_names) + 1) + "|"
    rows   = []
    for i, row in enumerate(cm):
        rows.append(f"| **actual:{class_names[i]}** | " + " | ".join(str(v) for v in row) + " |")

    lines = [
        "## 🤖 Model Evaluation Results",
        "",
        "| Metric | Value |",
        "|---|---|",
        f"| Accuracy | {accuracy * 100:.2f}% |",
        f"| Threshold | {threshold * 100:.0f}% |",
        f"| Status | {status} |",
        "",
        "### Confusion Matrix",
        "",
        header,
        sep,
        *rows,
        "",
        (
            "Model has been uploaded to GitHub Releases."
            if passed
            else "⚠️ Model did NOT meet the accuracy threshold — release skipped."
        ),
    ]
    return "\n".join(lines)


# Load results saved in Step 2
results = json.loads(pathlib.Path("/tmp/eval_results.json").read_text())
class_names = ["setosa", "versicolor", "virginica"]

comment_body = render_pr_comment(results, class_names)
print(comment_body)


**What just happened?**

- We generated the exact markdown that `actions/github-script` will post to the PR.
- The confusion matrix is a proper GitHub-flavoured Markdown table — renders beautifully in the PR UI.
- Testing the comment locally before wiring into CI saves debugging time — you can iterate on the format without pushing.


## Step 8 · Complete ML CI/CD Workflow (All Jobs)

Here is the full workflow combining test, train, evaluate (quality gate), release, and PR comment.


In [ ]:
# Complete ML CI/CD workflow — save to .github/workflows/ml-pipeline.yml
complete_ml_workflow = """\
name: ML CI/CD Pipeline

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    permissions:
      contents: read
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - name: Cache pip
        uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: ${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}
          restore-keys: ${{ runner.os }}-pip-
      - run: pip install -r requirements.txt
      - run: pytest tests/ -v

  train:
    needs: test
    runs-on: ubuntu-latest
    permissions:
      contents: read
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -r requirements.txt
      - name: Train model
        run: python train.py --output model.pkl
      - name: Upload model artifact
        uses: actions/upload-artifact@v4
        with:
          name: trained-model    # must match name in download steps exactly
          path: model.pkl
          retention-days: 7

  evaluate:
    needs: train
    runs-on: ubuntu-latest
    permissions:
      contents: read
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -r requirements.txt
      - name: Download model artifact
        uses: actions/download-artifact@v4
        with:
          name: trained-model
      - name: Evaluate (quality gate — fails if accuracy < 0.90)
        run: python evaluate.py --model model.pkl --threshold 0.90 --output eval_results.json
      - name: Upload evaluation results
        uses: actions/upload-artifact@v4
        with:
          name: eval-results
          path: eval_results.json

  release:
    needs: evaluate
    if: github.ref == 'refs/heads/main'  # only on main, not on PRs
    runs-on: ubuntu-latest
    permissions:
      contents: write   # needed to create a GitHub Release
    steps:
      - uses: actions/checkout@v4
      - name: Download model artifact
        uses: actions/download-artifact@v4
        with:
          name: trained-model
      - name: Upload to GitHub Releases
        uses: softprops/action-gh-release@v2  # pin to SHA in production
        with:
          tag_name: model-${{ github.sha }}
          name: Model ${{ github.sha }}
          files: model.pkl
          generate_release_notes: true

  comment:
    needs: evaluate
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    permissions:
      pull-requests: write  # needed to post a PR comment
      contents: read
    steps:
      - name: Download evaluation results
        uses: actions/download-artifact@v4
        with:
          name: eval-results
      - name: Post PR comment
        uses: actions/github-script@v7
        with:
          script: |
            const fs = require('fs');
            const r = JSON.parse(fs.readFileSync('eval_results.json', 'utf8'));
            const labels = ['setosa', 'versicolor', 'virginica'];
            const cmRows = r.confusion_matrix.map((row, i) =>
              `| **${labels[i]}** | ${row.join(' | ')} |`).join('\n');
            const body = [
              '## 🤖 Model Evaluation Results',
              `| Accuracy | ${(r.accuracy * 100).toFixed(2)}% |`,
              `| Threshold | ${(r.threshold * 100).toFixed(0)}% |`,
              `| Status | ${r.passed ? '✅ PASSED' : '❌ FAILED'} |`,
              '', '### Confusion Matrix', '',
              '| | pred:setosa | pred:versicolor | pred:virginica |',
              '|---|---|---|---|', cmRows,
            ].join('\n');
            await github.rest.issues.createComment({
              owner: context.repo.owner, repo: context.repo.repo,
              issue_number: context.issue.number, body,
            });
"""

print(complete_ml_workflow)

# Validate it's parseable YAML
parsed = yaml.safe_load(complete_ml_workflow)
print(f"\n✓ Valid YAML — {len(parsed['jobs'])} jobs: {list(parsed['jobs'].keys())}")


**What just happened?**

- We composed the full 5-job ML pipeline in one workflow file.
- `if: github.ref == 'refs/heads/main'` on the release job prevents model uploads from PR builds.
- `if: github.event_name == 'pull_request'` on the comment job ensures comments only go to PRs, not push events.
- **Both `release` and `comment` depend on `evaluate`** — if the quality gate fails, neither runs.


In [ ]:
# Challenge: Add a DVC data versioning step to the train job
#
# The pipeline above assumes data is in the repo. Real ML projects version
# large datasets with DVC (Data Version Control).
#
# Modify the train job steps to:
#   1. Add a step that runs: dvc pull  (pulls data from remote)
#      Hint: dvc pull needs credentials — pass them via env: using a secret
#   2. The DVC remote URL should be stored as a repo secret DVC_REMOTE_URL
#   3. Credentials (e.g. S3 key) should be stored as DVC_REMOTE_ACCESS_KEY
#   4. BONUS: Add a dvc repro step that re-runs the DVC pipeline instead of
#      calling train.py directly
#
# Your solution here:

train_job_with_dvc = {
    "train": {
        "needs": "test",
        "runs-on": "ubuntu-latest",
        "permissions": {"contents": "read"},
        "steps": [
            {"uses": "actions/checkout@v4"},
            {"uses": "actions/setup-python@v5", "with": {"python-version": "3.12"}},
            {"run": "pip install -r requirements.txt dvc"},
            # TODO: add dvc pull step with secrets passed via env
            # TODO: add train step (either python train.py or dvc repro)
            {
                "name": "Upload model artifact",
                "uses": "actions/upload-artifact@v4",
                "with": {"name": "trained-model", "path": "model.pkl"},
            },
        ],
    }
}

print(yaml.dump(train_job_with_dvc, sort_keys=False))


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| Artifact bridge | `upload-artifact` in train, `download-artifact` in evaluate and release — name must match exactly |
| Quality gate | `sys.exit(1)` in evaluate.py → job fails → release job skipped |
| `softprops/action-gh-release` | Upload `.pkl` to GitHub Releases — pin to SHA in production |
| `actions/github-script` | Inline JS to call GitHub API — post confusion matrix as PR comment |
| `if:` conditions | `github.ref == 'refs/heads/main'` and `github.event_name == 'pull_request'` gate per-job behaviour |
| `needs:` chain | test → train → evaluate → (release \| comment) |

> **Tip:** Use `softprops/action-gh-release` to upload model artifacts to GitHub Releases. Pin it to a commit SHA, not a tag, to prevent supply chain attacks.

---
## What's next
**Day 7** → Capstone — you'll build the complete ML CI/CD pipeline end-to-end: pytest, train, evaluate, release, and PR comment all in one production-ready workflow.

Mark Day 6 complete in your [tracker](../index.html).
